In [1]:
import platform
if 'mac' in platform.platform():
    BASE_DIR = "/Users/USER/vrtopc/"
    DATA_DIR = "/media/data/vrtopc"
else:
    BASE_DIR = "/home/USER/vr_to_pc/"
    DATA_DIR = "/media/data/vrtopc"

import sys
sys.path.append(BASE_DIR)

### Imports

In [2]:
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from utils.metrics import get_spatial_correlation

import scipy
import pandas as pd
import numpy as np
import os
import yaml

### Params

In [3]:
WITH_GRID_CELLS = True

# Processed data for the Science and Muessig papers

### Parameters

In [4]:
from utils.spatial_units import RateMaps, PolarMaps

N_SAMPLES_POS = RateMaps.N_SAMPLES_POS
PLACE_SI_TH = RateMaps.PLACE_SI_TH

N_SAMPLES_THET = PolarMaps.N_SAMPLES_THET
HD_SI_TH = PolarMaps.HD_SI_TH
HD_RVL_TH = PolarMaps.HD_RVL_TH


In [ ]:
edge_n_bins = 4

ONLY_2ND_TRIAL = True

AGES_TO_REMOVE = list(range(26, 32 +1))

SAVE_PLOTS = False
SAVE_DIR = None

### Load Data

In [6]:
ratnames_science = []
ratnames_muessig = []

data_dir = os.path.join(DATA_DIR, 'real_data', 'science2010_data_struct')
data = {}
for file in os.listdir(data_dir):
    if file.endswith('.mat'):
        name = file.split('.')[0].lower()
        if 'shuffled' in name:
            name += '_science2010'
        else:
            ratnames_science.append(name)
        data[name] = scipy.io.loadmat(os.path.join(data_dir, file))

data_dir = os.path.join(DATA_DIR, 'real_data', 'muessig_data_struct')
for file in os.listdir(data_dir):
    if file.endswith('.mat'):
        name = file.split('.')[0].lower()
        if 'shuffled' in name:
            name += '_muessig'
        else:
            ratnames_muessig.append(name)
            
        if name in data.keys():
            raise ValueError(f"Duplicate file name: {name}")
        data[name] = scipy.io.loadmat(os.path.join(data_dir, file))

sorted(list(data.keys()))[:5]

['r101_p20', 'r104_p26', 'r112_p40', 'r115_p24', 'r118_p24']

In [7]:
def get(x):
    return x[0][0]

data_dict = {}
ratnames_old = []

for k in data.keys():
    print(k)

    if not k.startswith('r'):
        print(f"File name {k} does not start with 'r', skipping")
        continue

    ratname = k
    
    if ratname not in data_dict.keys():
        data_dict[ratname] = {}
    
    d = get(data[k]['tmpS'])
    d_keys = list(d.dtype.names)

    dataset = d[d_keys.index('dataset')][0].split('_')[-1]
    ratnames_old.append(f"{ratname}_{dataset}")

    # sample rate is always 50 Hz
    ages = d[d_keys.index('age')][0] # age 40 denotes adult
    ages = [a if a<100 else 40 for a in ages]

    sample_rates = d[d_keys.index('sampleRate')][0]
    env_types = d[d_keys.index('envType')][0]
    ppm = d[d_keys.index('ppm')][0]
    spike_times = d[d_keys.index('spikeTimes')][0]
    is_cs_neuron = d[d_keys.index('isCSNeuron')][0]
    has_min_freq = d[d_keys.index('hasMinFreq')][0]
    pos = d[d_keys.index('positions')][0]
    hd = d[d_keys.index('directions')][0] # degrees
    speed = d[d_keys.index('speed')][0] # cm/s

    rate_maps = d[d_keys.index('rateMaps')][0]
    pos_occ = d[d_keys.index('posOccMaps')][0]
    si = d[d_keys.index('SI')][0]
    rate_maps_corr = d[d_keys.index('corrRateMaps')][0]
    si_corr = d[d_keys.index('SICorr')][0]
    rate_maps_hd8 = d[d_keys.index('rateMapsHD8')][0]
    rate_maps_hd4 = d[d_keys.index('rateMapsHD4')][0]
    
    polar_maps = d[d_keys.index('polarMaps')][0]
    si_pm = d[d_keys.index('dirSI')][0]
    rvl = d[d_keys.index('rvLength')][0]
    hd_occ = d[d_keys.index('dirOccMaps')][0]
    polar_maps_corr = d[d_keys.index('corrPolarMaps')][0]
    si_pm_corr = d[d_keys.index('dirSICorr')][0]
    rvl_corr = d[d_keys.index('rvLengthCorr')][0]
    polar_maps_pred = d[d_keys.index('predPolarMaps')][0]
    dis_ratios = d[d_keys.index('disRatios')][0]

    # there are always max 3 trials per day
    n_trials = 0
    for trial_n in range(len(ages)):
        if ONLY_2ND_TRIAL and (trial_n != 1) : continue # keep second trial

        rms = rate_maps[trial_n]
        sis = si[trial_n]
        rms_corr = rate_maps_corr[trial_n]
        sis_corr = si_corr[trial_n]
        rms_hd8 = rate_maps_hd8[trial_n]
        rms_hd4 = rate_maps_hd4[trial_n]

        pms = polar_maps[trial_n]
        sis_pm = si_pm[trial_n]
        rvls = rvl[trial_n]
        pms_corr = polar_maps_corr[trial_n]
        sis_pm_corr = si_pm_corr[trial_n]
        rvls_corr = rvl_corr[trial_n]
        drs = dis_ratios[trial_n]
        pms_pred = polar_maps_pred[trial_n]
        if rms.shape[-1] == 0 or pms.shape[-1] == 0:
            print(f"Skipping trial {trial_n} because rate maps or polar maps are empty")
            continue

        t = {}
        age = ages[trial_n]
        env = env_types[trial_n][0]
        p = pos[trial_n]
    
        if np.isnan(age) and (len(env) == 0) and (p.shape[-1] == 0):
            continue
        n_trials += 1

        age = int(age)
        if age not in data_dict[ratname].keys():
            data_dict[ratname][age] = {}
            data_dict[ratname][age]['trials'] = []

        t['name'] = n_trials
        t['environment'] = env
        t['ppm'] = ppm
        t['sample_rate'] = sample_rates[trial_n] # Hz
        t['positions'] = p
        t['x'] = p[:,0]
        t['y'] = p[:,1]
        if ratname in ratnames_science:
            t['spike_times'] = spike_times[trial_n][0]
            t['is_cs_neuron'] = is_cs_neuron[trial_n][0]
            t['has_min_freq'] = has_min_freq[trial_n][0]
        else:
            t['spike_times'] = spike_times[trial_n].squeeze() if len(spike_times[trial_n])>1 else spike_times[trial_n][0]
            t['is_cs_neuron'] = is_cs_neuron[trial_n].squeeze() if len(is_cs_neuron[trial_n])>1 else is_cs_neuron[trial_n][0]
            t['has_min_freq'] = has_min_freq[trial_n].squeeze() if len(has_min_freq[trial_n])>1 else has_min_freq[trial_n][0]

        t['speed'] = speed[trial_n].squeeze()/100 # m/s
        t['hd'] = hd[trial_n].squeeze()
        t['duration'] = len(t['x'])/t['sample_rate']

        # convert to (n_units, n_bins, n_bins)
        t['rate_maps'] = np.array([rms[idx][0] for idx in range(len(rms))])
        t['si'] = np.array([sis[idx][0] for idx in range(len(sis))])
        t['rate_maps_corr'] = np.array([rms_corr[idx][0] for idx in range(len(rms_corr))])
        t['si_corr'] = np.array([sis_corr[idx][0] for idx in range(len(sis_corr))])

        rms_hd_np8 = np.zeros((t['rate_maps'].shape[0], 8, t['rate_maps'].shape[-1], t['rate_maps'].shape[-1]))
        rms_hd_np4 = np.zeros((t['rate_maps'].shape[0], 4, t['rate_maps'].shape[-1], t['rate_maps'].shape[-1]))
        for j in range(8):
            if j < 4:
                rms_hd_np4[:, j, ...] = np.array(
                    [get(rms_hd4[idx])[j] for idx in range(len(rms_hd4))]
                )
            rms_hd_np8[:, j, ...] = np.array(
                [get(rms_hd8[idx])[j] for idx in range(len(rms_hd8))]
            )
        # convert to (n_units, 8, n_bins, n_bins)
        t['rate_maps_hd8'] = rms_hd_np8
        t['rate_maps_hd4'] = rms_hd_np4

        # convert to (n_units, n_bins)
        t['polar_maps'] = np.array([pms[idx][0] for idx in range(len(pms))])[..., 0]
        t['si_pm'] = np.array([sis_pm[idx][0] for idx in range(len(sis_pm))])
        t['rvl'] = np.array([rvls[idx][0] for idx in range(len(rvls))])
        t['polar_maps_corr'] = np.array([pms_corr[idx][0] for idx in range(len(pms_corr))])[..., 0]
        t['si_pm_corr'] = np.array([sis_pm_corr[idx][0] for idx in range(len(sis_pm_corr))])
        t['rvl_corr'] = np.array([rvls_corr[idx][0] for idx in range(len(rvls_corr))])

        t['polar_maps_pred'] = np.array([pms_pred[idx][0] for idx in range(len(pms_pred))])[..., 0]
        t['dis_ratios'] = np.array(drs[:,0])

        if t['rate_maps'].shape[0] != t['polar_maps'].shape[0]:
            raise ValueError(f"Rate maps ({t['rate_maps'].shape}) and polar maps ({t['polar_maps'].shape}) have different number of units")

        t['pos_occ'] = get(pos_occ[trial_n])
        t['hd_occ'] = np.array(hd_occ[trial_n][:,0])

        data_dict[ratname][age]['trials'].append(t)
    print(f"\t{n_trials} trial(s)")
    print()

r1308_d4
	1 trial(s)

r1526_p18
	1 trial(s)

r1343_d4
	1 trial(s)

r1526_p23
	1 trial(s)

r1343_d1
	1 trial(s)

r1589_p22
	1 trial(s)

r1589_p28
	1 trial(s)

r1333_d1
	1 trial(s)

r1477_p29
	1 trial(s)

r1552_p22
	1 trial(s)

r1637_p23
	1 trial(s)

r1588_p20
	1 trial(s)

r1589_p23
	1 trial(s)

r1589_p27
	1 trial(s)

r1588_p22
	1 trial(s)

r1515_p23
	1 trial(s)

r1589_p17
	1 trial(s)

r1589_p26
	1 trial(s)

r1552_p17
	1 trial(s)

r1308_d1
	1 trial(s)

r1526_p20
	1 trial(s)

r1589_p21
	1 trial(s)

r1552_p16_1
	1 trial(s)

r1589_p25
	1 trial(s)

r1333_d2
	1 trial(s)

r1628_p22_ca1
	1 trial(s)

r1589_p19
	1 trial(s)

r1515_p22
	1 trial(s)

r1526_p21
	1 trial(s)

r1588_p24
	1 trial(s)

r1474_p25
	1 trial(s)

r1552_p16_2
	1 trial(s)

r1588_p16
	1 trial(s)

shuffled_metrics_science2010
File name shuffled_metrics_science2010 does not start with 'r', skipping
shuffled_metrics_adult_science2010
File name shuffled_metrics_adult_science2010 does not start with 'r', skipping
r1588_p21
	1 trial(s)



In [8]:
ages = []
print('--------------------------------')
for k, v in data_dict.items():
    print(k)
    ages_tmp = list(v.keys())
    print('ages', ages_tmp)
    ages += ages_tmp
    print()
    print('--------------------------------')

--------------------------------
r1308_d4
ages [40]

--------------------------------
r1526_p18
ages [18]

--------------------------------
r1343_d4
ages [40]

--------------------------------
r1526_p23
ages [23]

--------------------------------
r1343_d1
ages [40]

--------------------------------
r1589_p22
ages [22]

--------------------------------
r1589_p28
ages [28]

--------------------------------
r1333_d1
ages [40]

--------------------------------
r1477_p29
ages [29]

--------------------------------
r1552_p22
ages [22]

--------------------------------
r1637_p23
ages [23]

--------------------------------
r1588_p20
ages [20]

--------------------------------
r1589_p23
ages [23]

--------------------------------
r1589_p27
ages [27]

--------------------------------
r1588_p22
ages [22]

--------------------------------
r1515_p23
ages [23]

--------------------------------
r1589_p17
ages [17]

--------------------------------
r1589_p26
ages [26]

--------------------------------

### Shuffled Threshold Extraction

In [9]:
metrics_shuffle_th_science = {}
metrics_shuffle_th_muessig = {}

for filename in data.keys():
    if not filename.startswith('shuffled'):
        continue
    print(filename)

    shuffled_si = data[filename]['shuffledSIByAge']
    shuffled_si_pm = data[filename]['shuffledDirSIByAge']
    shuffled_rvl = data[filename]['shuffledRVLByAge']
    for idx in range(len(shuffled_si)):
        for m, k in zip([shuffled_si, shuffled_si_pm, shuffled_rvl], ['SI', 'dirSI', 'rvLength']):
            m_curr = get(m[idx][0])
            keys = list(m_curr.dtype.names)
            age_group = get(m_curr[keys.index('ageGroup')])
            th = get(m_curr[keys.index(k+'Threshold')])

            if 'science' in filename.lower():
                if k not in metrics_shuffle_th_science.keys():
                    metrics_shuffle_th_science[k] = {}

                age = 14+age_group*2
                if age != 100:
                    metrics_shuffle_th_science[k][age] = th
                    metrics_shuffle_th_science[k][age+1] = th
                else : metrics_shuffle_th_science[k][40] = th
            elif 'muessig' in filename.lower():
                if k not in metrics_shuffle_th_muessig.keys():
                    metrics_shuffle_th_muessig[k] = {}

                age = 14+age_group*2
                if age != 100:
                    metrics_shuffle_th_muessig[k][age] = th
                    metrics_shuffle_th_muessig[k][age+1] = th
                else : metrics_shuffle_th_muessig[k][40] = th


shuffled_metrics_science2010
shuffled_metrics_adult_science2010
shuffled_metrics_muessig


### Activity Extraction

A neuron is considered actually tuned to direction if its corrected polar map still passes the criterion for inclusion (RVL or KLD)

and its Pearson correlation with the uncorrected polar map is higher than 0.5

In [10]:
def normalize_rate_maps(rate_maps):
    # normalize rate maps
    rate_maps_min = np.moveaxis(
        np.tile(np.nanmin(rate_maps, axis=(1,2)), (N_SAMPLES_POS, N_SAMPLES_POS, 1)), -1, 0
    )
    rate_maps_max = np.moveaxis(
        np.tile(np.nanmax(rate_maps, axis=(1,2)), (N_SAMPLES_POS, N_SAMPLES_POS, 1)), -1, 0
    )
    rate_maps = (
        (rate_maps - rate_maps_min) / (rate_maps_max - rate_maps_min)
    )
    return rate_maps

In [11]:
from utils.spatial_units import RateMaps, PolarMaps

place_units = RateMaps(positions=None, env_dim=0)
hd_units = PolarMaps(thetas=None)

data_dict_age = {}
perc_kept = []

for k, v in sorted(data_dict.items()):
    print(f"Rat {k}")
    if k in ratnames_science:
        metrics_shuffle_th = metrics_shuffle_th_science
    elif k in ratnames_muessig:
        metrics_shuffle_th = metrics_shuffle_th_muessig
    else:
        raise ValueError(f"Rat {k} not found in science or muessig data")
    
    for age in sorted(v.keys()):
        if age in AGES_TO_REMOVE:
            print(f"\tAge {age} in ages to remove, skipping")
            continue
        
        exp = v[age] # get the experiment for this rat's age
        print(f"\tAge {age}")
        if (age in data_dict_age.keys()) and (k in data_dict_age[age].keys()):
            print(f"\tSkipping because already processed")
            continue

        if age not in data_dict_age.keys(): # initialize all dict if first exp for this age
            data_dict_age[age] = {}
        
        data_dict_age[age][k] = {}
        for k_tmp in [
            'positions', 'hd', 'speed', 'spike_times', 'sample_rate',
            'rate_maps', 'pos_occ', 'rate_maps_corr',
            'si_matlab', 'si_corr_matlab', 'si_rm', 'si_rm_corr',
            'selected_place_units', 'n_fields',
            'single_field_dim', 'pu_flipped', 'pu_field_flipped',
            'rate_maps_hd8', 'rate_maps_hd4',
            'polar_maps', 'hd_occ', 'polar_maps_corr',
            'si_pm_matlab', 'rvl_matlab', 'si_pm', 'rvl_pm', 'si_pm_corr_matlab', 'rvl_corr_matlab',
            'rvl_pm_corr', 'rvangle_pm', 'rvangle_pm_corr',
            'selected_hd_units', 'selected_place_hd_units',
            'polar_maps_pred', 'dis_ratios',
            'trial_start_idx'
        ]:
            data_dict_age[age][k][k_tmp] = []

        rate_maps_all = []
        polar_maps_all = []
        indices_to_keep = None
        trial_start_idx = 0
        for trial in exp['trials']:
            if trial['environment'] != 'hp' and trial['environment'] != 'fam':
                raise ValueError(f"\tEnvironment is {trial['environment']} instead of hp or fam")

            rate_maps = trial['rate_maps']
            if rate_maps.shape[1] != N_SAMPLES_POS or rate_maps.shape[2] != N_SAMPLES_POS:
                raise ValueError(f"\t\tRate maps shape is {rate_maps.shape} instead of (n_cells, {N_SAMPLES_POS}, {N_SAMPLES_POS})")
            rate_maps_all.append(rate_maps.copy())

            si_matlab = trial['si']
            rate_maps_corr = trial['rate_maps_corr']
            si_corr_matlab = trial['si_corr']
            rate_maps_hd8 = trial['rate_maps_hd8']
            rate_maps_hd4 = trial['rate_maps_hd4']

            pos_occ = trial['pos_occ']

            polar_maps = trial['polar_maps']
            if polar_maps.shape[1] != N_SAMPLES_THET:
                raise ValueError(f"\t\Polar maps shape is {polar_maps.shape} instead of (n_cells, {N_SAMPLES_THET})")
            polar_maps_all.append(polar_maps.copy())

            si_pm_matlab = trial['si_pm']
            rvl_matlab = trial['rvl']
            polar_maps_corr = trial['polar_maps_corr']
            si_pm_corr_matlab = trial['si_pm_corr']
            rvl_corr_matlab = trial['rvl_corr']
            polar_maps_pred = trial['polar_maps_pred']
            dis_ratios = trial['dis_ratios']

            hd_occ = trial['hd_occ']

            print(f"\t(n_cells, n_samples_pos, n_samples_pos): {rate_maps.shape}")
            print(f"\t(n_cells, N_SAMPLES_THET): {polar_maps.shape}")

            # keep only Complex Spike neurons
            idx_to_keep = np.logical_and(
                trial['is_cs_neuron'] == 1, trial['has_min_freq'] == 1
            )
            if isinstance(idx_to_keep, np.bool) : idx_to_keep = np.array([idx_to_keep])
            assert len(idx_to_keep) == rate_maps.shape[0]
            idx_to_keep = np.where(idx_to_keep)[0] # convert mask to indices
            if len(idx_to_keep) == 0:
                print(f"\tSkipping trial because all rate maps are uniform")
                continue

            positions = trial['positions']
            hd = trial['hd'].astype(np.float64)
            speed = trial['speed']
            spike_times = trial['spike_times']
            sample_rate = trial['sample_rate']
            
            perc_kept.append(len(idx_to_keep)/rate_maps.shape[0]*100)
            rate_maps = rate_maps[idx_to_keep]
            si_matlab = si_matlab[idx_to_keep]
            rate_maps_corr = rate_maps_corr[idx_to_keep]
            si_corr_matlab = si_corr_matlab[idx_to_keep]
            rate_maps_hd8 = rate_maps_hd8[idx_to_keep]
            rate_maps_hd4 = rate_maps_hd4[idx_to_keep]

            polar_maps = polar_maps[idx_to_keep]
            si_pm_matlab = si_pm_matlab[idx_to_keep]
            rvl_matlab = rvl_matlab[idx_to_keep]
            polar_maps_corr = polar_maps_corr[idx_to_keep]
            si_pm_corr_matlab = si_pm_corr_matlab[idx_to_keep]
            rvl_corr_matlab = rvl_corr_matlab[idx_to_keep]
            polar_maps_pred = polar_maps_pred[idx_to_keep]
            dis_ratios = dis_ratios[idx_to_keep]

            if indices_to_keep is None: indices_to_keep = idx_to_keep
            else: indices_to_keep = np.intersect1d(indices_to_keep, idx_to_keep)
            
            rate_maps_unnorm = rate_maps.copy()
            rate_maps_corr_unnorm = rate_maps_corr.copy()
            rate_maps = normalize_rate_maps(rate_maps)
            rate_maps_corr = normalize_rate_maps(rate_maps_corr)
            
            si_rm = place_units.calculate_metrics(rate_maps, pos_occ)
            si_rm_corr = place_units.calculate_metrics(rate_maps_corr, pos_occ)
            
            n_fields, rm_fields = place_units.rate_maps_field_detection(rate_maps, rate_maps, rate_maps)

            selected_place_units = place_units.get_place_cells_indices(rate_maps, si_matlab)

            if len(selected_place_units) > 0:
                single_field_dim = np.array([
                    np.sum(np.nansum(np.array(fields), axis=0)>0) for i, fields in enumerate(rm_fields)
                    if fields and i in selected_place_units
                ])
                pu_flipped = place_units.rm_flipped(rate_maps, filter_indices=selected_place_units)

                rm_fields_selected = [f for i, f in enumerate(rm_fields) if (i in selected_place_units) and n_fields[i] > 0]
                if len(rm_fields_selected) == 0:
                    print(f"\tSkipping avg rate map field because no selected fields")
                    continue
                pu_field_flipped = place_units.rm_field_flipped(rm_fields_selected)

                for k_tmp in ['single_field_dim', 'pu_flipped', 'pu_field_flipped']:
                    data_dict_age[age][k][k_tmp].append(locals()[k_tmp])
            
            si_pm, rvl_pm, rvangle_pm = hd_units.calculate_metrics(polar_maps.copy(), hd_occ)
            _, rvl_pm_corr, rvangle_pm_corr = hd_units.calculate_metrics(polar_maps_corr.copy(), hd_occ)

            selected_hd_units = np.array([
                idx for idx in range(polar_maps.shape[0]) if
                (not np.isnan(si_pm_matlab[idx])) and (not np.isnan(rvl_matlab[idx])) and
                (not np.isnan(si_pm_corr_matlab[idx])) and (not np.isnan(rvl_corr_matlab[idx])) and
                ((si_pm_matlab[idx] > metrics_shuffle_th['dirSI'][age]) or (rvl_matlab[idx] > metrics_shuffle_th['rvLength'][age])) and
                ((si_pm_corr_matlab[idx] > metrics_shuffle_th['dirSI'][age]) or (rvl_corr_matlab[idx] > metrics_shuffle_th['rvLength'][age])) and
                (get_spatial_correlation(polar_maps[idx], polar_maps_corr[idx], return_pvalue=False) > 0.5)# and
            ], dtype=np.int32)

            selected_place_hd_units = np.intersect1d(selected_place_units, selected_hd_units, assume_unique=True)
            selected_place_units = np.setdiff1d(selected_place_units, selected_place_hd_units, assume_unique=True)
            selected_hd_units = np.setdiff1d(selected_hd_units, selected_place_hd_units, assume_unique=True)
            
            selected_place_units += trial_start_idx
            selected_hd_units += trial_start_idx
            selected_place_hd_units += trial_start_idx

            for k_tmp in [
                'positions', 'hd', 'speed', 'spike_times', 'sample_rate',
                'rate_maps', 'rate_maps_corr', 'pos_occ',
                'si_matlab', 'si_corr_matlab', 'si_rm', 'si_rm_corr',
                'selected_place_units', 'n_fields',
                'rate_maps_hd8', 'rate_maps_hd4',
                'polar_maps', 'polar_maps_corr', 'hd_occ',
                'si_pm_matlab', 'rvl_matlab', 'si_pm', 'rvl_pm', 'si_pm_corr_matlab', 'rvl_corr_matlab',
                'rvl_pm_corr', 'rvangle_pm', 'rvangle_pm_corr',
                'selected_hd_units', 'selected_place_hd_units',
                'polar_maps_pred', 'dis_ratios',
                'trial_start_idx'
            ]:
                data_dict_age[age][k][k_tmp].append(locals()[k_tmp])

            trial_start_idx += len(idx_to_keep)

        if trial_start_idx == 0:
            data_dict_age[age].pop(k)
        else:

            for k_tmp in [
                'rate_maps', 'rate_maps_corr', 'pos_occ',
                'si_matlab', 'si_corr_matlab', 'si_rm', 'si_rm_corr',
                'selected_place_units', 'n_fields',
                'rate_maps_hd8', 'rate_maps_hd4',
                'polar_maps', 'polar_maps_corr', 'hd_occ',
                'si_pm_matlab', 'rvl_matlab', 'si_pm', 'rvl_pm', 'si_pm_corr_matlab', 'rvl_corr_matlab',
                'rvl_pm_corr', 'rvangle_pm', 'rvangle_pm_corr',
                'selected_hd_units', 'selected_place_hd_units',
                'polar_maps_pred', 'dis_ratios',
            ]:
                try:
                    data_dict_age[age][k][k_tmp] = np.concatenate(data_dict_age[age][k][k_tmp])
                except ValueError as e:
                    if "zero-dimensional" in str(e):
                        data_dict_age[age][k][k_tmp] = np.array(data_dict_age[age][k][k_tmp])
                    elif "need at least one" in str(e):
                        continue
                    else:
                        raise e
            
    print(flush=True)

Rat r101_p20
	Age 20
	(n_cells, n_samples_pos, n_samples_pos): (8, 25, 25)
	(n_cells, N_SAMPLES_THET): (8, 60)

Rat r104_p26
	Age 26 in ages to remove, skipping



Rat r112_p40
	Age 40
	(n_cells, n_samples_pos, n_samples_pos): (27, 25, 25)
	(n_cells, N_SAMPLES_THET): (27, 60)

Rat r115_p24
	Age 24
	(n_cells, n_samples_pos, n_samples_pos): (9, 25, 25)
	(n_cells, N_SAMPLES_THET): (9, 60)

Rat r118_p24
	Age 24
	(n_cells, n_samples_pos, n_samples_pos): (21, 25, 25)
	(n_cells, N_SAMPLES_THET): (21, 60)

Rat r118_p25
	Age 25
	(n_cells, n_samples_pos, n_samples_pos): (11, 25, 25)
	(n_cells, N_SAMPLES_THET): (11, 60)

Rat r118_p26
	Age 26 in ages to remove, skipping

Rat r118_p27
	Age 27 in ages to remove, skipping

Rat r1262_d1
	Age 40
	(n_cells, n_samples_pos, n_samples_pos): (13, 25, 25)
	(n_cells, N_SAMPLES_THET): (13, 60)

Rat r1262_d3
	Age 40
	(n_cells, n_samples_pos, n_samples_pos): (12, 25, 25)
	(n_cells, N_SAMPLES_THET): (12, 60)

Rat r129_p40
	Age 40
	(n_cells, n_samples_pos, n_samples_pos): (9, 25, 25)
	(n_cells, N_SAMPLES_THET): (9, 60)

Rat r1308_d1
	Age 40
	(n_cells, n_samples_pos, n_samples_pos): (14, 25, 25)
	(n_cells, N_SAMPLES_THET): (1

In [12]:
ages = sorted(data_dict_age.keys())
n_ages = len(ages)

In [13]:
si_rm_selected = {}
si_pm_selected = {}
rvl_pm_selected = {}

for age in ages:
    for rat in data_dict_age[age].keys():
        si_rm = data_dict_age[age][rat]['si_rm']
        si_pm = data_dict_age[age][rat]['si_pm']
        rvl_pm = data_dict_age[age][rat]['rvl_pm']

        selected_place_units = data_dict_age[age][rat]['selected_place_units']
        selected_hd_units = data_dict_age[age][rat]['selected_hd_units']
        selected_place_hd_units = data_dict_age[age][rat]['selected_place_hd_units']

        selected_units_rm = np.concatenate([selected_place_units, selected_place_hd_units])
        selected_units_pm = np.concatenate([selected_hd_units, selected_place_hd_units])

        if age not in si_rm_selected.keys():
            si_rm_selected[age] = []
            si_pm_selected[age] = []
            rvl_pm_selected[age] = []
        si_rm_selected[age].append(si_rm[selected_units_rm])
        si_pm_selected[age].append(si_pm[selected_units_pm])
        rvl_pm_selected[age].append(rvl_pm[selected_units_pm])

for age in si_pm_selected.keys():
    si_rm_selected[age] = np.concatenate(si_rm_selected[age])
    si_pm_selected[age] = np.concatenate(si_pm_selected[age])
    rvl_pm_selected[age] = np.concatenate(rvl_pm_selected[age])

    print(f"Age {age}:")
    if len(si_rm_selected[age]) > 0:
        print(f"\tPlace units SI: {np.nanmean(si_rm_selected[age]):.3f} (min {np.min(si_rm_selected[age]):.3f})")
    if len(rvl_pm_selected[age]) > 0:
        print(f"\tHD units RVL: {np.nanmean(rvl_pm_selected[age]):.3f} (min {np.min(rvl_pm_selected[age]):.3f})")
        print(f"\tHD units SI: {np.nanmean(si_pm_selected[age]):.3f} (min {np.min(si_pm_selected[age]):.3f})")
    print(flush=True)

Age 14:
	Place units SI: 0.607 (min 0.393)
	HD units RVL: 0.317 (min 0.292)
	HD units SI: 0.624 (min 0.539)

Age 15:
	Place units SI: 0.619 (min 0.369)
	HD units RVL: 0.227 (min 0.174)
	HD units SI: 0.303 (min 0.237)

Age 16:
	Place units SI: 0.814 (min 0.372)
	HD units RVL: 0.355 (min 0.245)
	HD units SI: 0.372 (min 0.208)

Age 17:
	Place units SI: 0.662 (min 0.365)
	HD units RVL: 0.329 (min 0.066)
	HD units SI: 0.334 (min 0.113)

Age 18:
	Place units SI: 0.657 (min 0.353)
	HD units RVL: 0.344 (min 0.047)
	HD units SI: 0.321 (min 0.134)

Age 19:
	Place units SI: 0.745 (min 0.346)
	HD units RVL: 0.352 (min 0.060)
	HD units SI: 0.322 (min 0.146)

Age 20:
	Place units SI: 0.747 (min 0.364)
	HD units RVL: 0.352 (min 0.113)
	HD units SI: 0.360 (min 0.139)

Age 21:
	Place units SI: 0.753 (min 0.370)
	HD units RVL: 0.392 (min 0.271)
	HD units SI: 0.356 (min 0.133)

Age 22:
	Place units SI: 0.783 (min 0.363)
	HD units RVL: 0.340 (min 0.158)
	HD units SI: 0.298 (min 0.131)

Age 23:
	Place unit

# Load clustering data

#### Params

In [14]:
BY = 'day'
SEED = 7
CLUSTERALGO = 'gm'

#### Loading

In [15]:
df_dir = os.path.join(DATA_DIR, 'cluster_locomotion', f'by_{BY}')
df_data = pd.read_pickle(os.path.join(df_dir, f'data_{SEED}.pkl'))

c_idx_col = f'cluster_idx_{CLUSTERALGO}'
df_data = df_data[df_data[c_idx_col] != -1].reset_index()
df_data.loc[df_data['age'] == 100, 'age'] = 40

In [16]:
import re

if 'exp' in BY or 'day' in BY:
    df_data['dataset'] = df_data['rat'].map(lambda x: re.sub("['() ]", "", x.split(',')[0]))
    df_data = df_data[
        (df_data['dataset'].str.contains('science2010')) | (df_data['dataset'].str.contains('muessig'))
    ]
    df_data['rat'] = df_data['rat'].map(lambda x: re.sub("['() ]", "", x.split(',')[1]))

    # this dataset give us a cluster for each (rat, age, trial)
    # we want to get a cluster for each (rat, age)
    # we exclude the (rat, age) where there are multiple clusters
    df_data_exclude = df_data.groupby(['dataset', 'rat', 'age']).agg({c_idx_col: 'nunique'})
    df_data_exclude = df_data_exclude[df_data_exclude[c_idx_col] > 1].reset_index()

    print(
        f"Excluding {len(df_data_exclude)/len(df_data.reset_index().groupby(['rat', 'age']).count())*100:.1f}% "+
        "(rat, age) pairs because they have multiple clusters"
    )

    df_merge = df_data.merge(df_data_exclude[['dataset', 'rat', 'age']], on=['dataset', 'rat', 'age'], how='left', indicator=True)
    df_data = df_merge[df_merge['_merge'] == 'left_only'].drop(columns=['_merge'])

    # keep first cluster index because they are all the same after previous operation
    df_data = df_data.groupby(['dataset', 'rat', 'age']).agg({c_idx_col: lambda x: list(x)[0]})
elif 'age' in BY:
    df_data = df_data[['age', c_idx_col]]
    
df_data.head()

Excluding 0.0% (rat, age) pairs because they have multiple clusters


cluster_idx_gm
dataset             rat      age                
muessig_data_struct r101_p20 20                2
                    r115_p24 24                2
                    r118_p24 24                2
                    r118_p25 25                2
                    r129_p40 40                3

In [17]:
df_data = df_data.reset_index()


In [ ]:
df_data_merge = df_data.groupby(c_idx_col).agg({'rat': 'count', 'age': 'median'}).reset_index()
df_data_merge = df_data_merge[df_data_merge['rat'] > 2]
df_data = pd.merge(
    df_data, df_data_merge[c_idx_col], on=c_idx_col, how='inner'
)
df_data.groupby(c_idx_col).agg({'rat': 'count', 'age': 'median'})


,rat,age
cluster_idx_gm,,
1,31,16.0
2,102,20.0
3,9,40.0


In [21]:
clusters = [c if c < 3 else 'Adult' for c in sorted(df_data[c_idx_col].unique())]

# DEV model data

### Parameters

In [ ]:
args_dev = { # DEVELOPMENT
    'behaviour' : ['crawl', 'walk', 'run', 'adult', 'adult'],
    'pretrained_behav' : ['crawl', 'crawl,walk', 'crawl,walk,run', 'crawl,walk,run,adult', 'adult'],
    'env' : 'box_messy',
    'env_dim': 0.635,
    'name_prefix': None,
    'pretrained_model_folder': False,
    'moredata': None,
    'n_gridcells': [0,0,0,0,25], # with GC
    'gridcells_softmax': [False,False,False,False,True], # with GC
    'gridcells_modules': [None,None,None,None,[0.2,0.4,0.6]], # with GC
    'gridcells_orientations': [None,None,None,None,[0.1]], # with GC
    'n_future_pred' : 1,
    'frame_subsampling': 4,
    'stride' : 10,
    'reset_hidden_at': [None,None,None,None,10], # with GC
    'bptt_steps' : 9,
    'latent_dim' : 500,
    'bias': False,
    'dropouts': '[0,0,0]',
    'nonlinearity' : 'sigmoid',
    'hidden_reg' : 0.,
    'weights_reg' : 0.,
    'seed': 1,
    'epoch' : None,
}

### Define Data

In [23]:
n_compare = None
print("Comparison will be done on the following parameters:")
for k, a in args_dev.items():
    if isinstance(a, list):
        print(f"\t{k}: {a}")
        if n_compare is None:
            n_compare = len(a)
        elif n_compare != len(a):
            raise ValueError("All lists must have the same length")

if n_compare is None:
    raise ValueError("At least one argument must be a list to make a comparison")

for k, a in args_dev.items():
    if not isinstance(a, list):
        args_dev[k] = [a] * n_compare

Comparison will be done on the following parameters:
	behaviour: ['crawl', 'walk', 'run', 'adult', 'adult']
	pretrained_behav: ['crawl', 'crawl,walk', 'crawl,walk,run', 'crawl,walk,run,adult', 'adult']
	n_gridcells: [0, 0, 0, 0, 25]
	gridcells_softmax: [False, False, False, False, True]
	gridcells_modules: [None, None, None, None, [0.2, 0.4, 0.6]]
	gridcells_orientations: [None, None, None, None, [0.1]]
	reset_hidden_at: [None, None, None, None, 10]


In [24]:
from utils.trainer import RNNTrainer

activity_dirs_dev = []
models_dev = []

print("Comparing the activity from the following directories:")
for i in range(n_compare):
    model_name = RNNTrainer.define_model_name({k: v[i] for k, v in args_dev.items()})
    
    env_shape = args_dev['env'][i].split('_')[0]
    trained_behav_list = args_dev['pretrained_behav'][i].split(',')
    behav = trained_behav_list.pop()
    if len(trained_behav_list)>0:
        folder_name = '_'.join(trained_behav_list)
    else:
        folder_name = "vanilla"
    exp_dir = os.path.join(
        DATA_DIR, env_shape, behav, "predictions", args_dev['env'][i],
        folder_name, model_name
    )

    activity_dir = f"act_{args_dev['behaviour'][i]}_epoch"
    if args_dev['epoch'][i] is not None:
        epoch = args_dev['epoch'][i]
    else:
        dirs = [d for d in os.listdir(exp_dir) if re.match(rf"{activity_dir}\d+", d)]
        epoch = max([int(re.findall(r'\d+', d)[-1]) for d in dirs])

    models_dev.append(
        torch.load(
            os.path.join(exp_dir, f"rnn_epoch{epoch}.pth"),
            weights_only=False,
            map_location=torch.device(DEVICE)
        ).to(DEVICE)
    )
    activity_dirs_dev.append(os.path.join(exp_dir, f"{activity_dir}{epoch}"))

    print(activity_dirs_dev[-1])

Comparing the activity from the following directories:
/media/data/vrtopc/box/crawl/predictions/box_messy/vanilla/RNN_f1_w9_st10_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_crawl_epoch1500
/media/data/vrtopc/box/walk/predictions/box_messy/crawl/RNN_f1_w9_st10_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_walk_epoch1500
/media/data/vrtopc/box/run/predictions/box_messy/crawl_walk/RNN_f1_w9_st10_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_run_epoch1500
/media/data/vrtopc/box/adult/predictions/box_messy/crawl_walk_run/RNN_f1_w9_st10_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_adult_epoch1500
/media/data/vrtopc/box/adult/predictions/box_messy/vanilla/RNN_gridcellssm25_mod[0.2,0.4,0.6]_ori[0.1]_reset10_f1_w9_st10_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_adult_epoch1500


# ROC model data

### Parameters

In [ ]:
args_compare = (
    # ###################################################################################
    # ########################## DIS-AGREEING MODELS ####################################
    # ###################################################################################
    { # RATE OF CHANGE
        'behaviour' : ['crawl', 'crawl', 'crawl', 'crawl', 'crawl'],
        'pretrained_behav' : ['crawl', 'crawl', 'crawl', 'crawl', 'crawl'],
        'env' : 'box_messy',
        'env_dim': 0.635,
        'name_prefix': None,
        'pretrained_model_folder': [False,True,True,True,False],
        'moredata': None,
        'n_gridcells': [0,0,0,0,25], # with GC
        'gridcells_softmax': [False,False,False,False,True], # with GC
        'gridcells_modules': [None,None,None,None,[0.2,0.4,0.6]], # with GC
        'gridcells_orientations': [None,None,None,None,[0.1]], # with GC
        'n_future_pred' : 1,
        'frame_subsampling': 4,
        'stride' : [10, 20, 25, 30, 30],
        'reset_hidden_at': [None,None,None,None,10], # with GC
        'bptt_steps' : 9,
        'latent_dim' : 500,
        'bias': False,
        'dropouts': '[0,0,0]',
        'nonlinearity' : 'sigmoid',
        'hidden_reg' : 0.,
        'weights_reg' : 0.,
        'seed': 1,
        'epoch' : None,
    }
    # { # CRAWL WITH MORE DATA
    #     'behaviour' : ['crawl', 'crawl', 'crawl', 'crawl', 'crawl'],
    #     'pretrained_behav' : ['crawl', 'crawl', 'crawl', 'crawl', 'crawl'],
    #     'env' : 'box_messy',
    #     'env_dim': 0.635,
    #     'name_prefix': None,
    #     'pretrained_model_folder': [False,True,True,True,False],
    #     'moredata': [None, 1, 2, 3, 3],
    #     'n_gridcells': [0,0,0,0,25], # with GC
    #     'gridcells_softmax': [False,False,False,False,True], # with GC
    #     'gridcells_modules': [None,None,None,None,[0.2,0.4,0.6]], # with GC
    #     'gridcells_orientations': [None,None,None,None,[0.1]], # with GC
    #     'n_future_pred' : 1,
    #     'frame_subsampling': 4,
    #     'stride' : 10,
    #     'reset_hidden_at': [None,None,None,None,10], # with GC
    #     'bptt_steps' : 9,
    #     'latent_dim' : 500,
    #     'bias': False,
    #     'dropouts': '[0,0,0]',
    #     'nonlinearity' : 'sigmoid',
    #     'hidden_reg' : 0.,
    #     'weights_reg' : 0.,
    #     'seed': 1,
    #     'epoch' : None,
    # }
    # { # REVERSE TRAINING
    #     'behaviour' : ['adult', 'run', 'walk', 'crawl', 'crawl'],
    #     'pretrained_behav' : ['adult', 'adult,run', 'adult,run,walk', 'adult,run,walk,crawl', 'crawl'],
    #     'env' : 'box_messy',
    #     'env_dim': 0.635,
    #     'name_prefix': None,
    #     'pretrained_model_folder': False,
    #     'moredata': None,
    #     'n_gridcells': [0,0,0,0,25], # with GC
    #     'gridcells_softmax': [False,False,False,False,True], # with GC
    #     'gridcells_modules': [None,None,None,None,[0.2,0.4,0.6]], # with GC
    #     'gridcells_orientations': [None,None,None,None,[0.1]], # with GC
    #     'n_future_pred' : 1,
    #     'frame_subsampling': 4,
    #     'stride' : 10,
    #     'reset_hidden_at': [None,None,None,None,10], # with GC
    #     'bptt_steps' : 9,
    #     'latent_dim' : 500,
    #     'bias': False,
    #     'dropouts': '[0,0,0]',
    #     'nonlinearity' : 'sigmoid',
    #     'hidden_reg' : 0.,
    #     'weights_reg' : 0.,
    #     'seed': 1,
    #     'epoch' : None,
    # }
)

### Define Data

In [26]:
n_compare = None
print("Comparison will be done on the following parameters:")
for k, a in args_compare.items():
    if isinstance(a, list):
        print(f"\t{k}: {a}")
        if n_compare is None:
            n_compare = len(a)
        elif n_compare != len(a):
            raise ValueError("All lists must have the same length")

if n_compare is None:
    raise ValueError("At least one argument must be a list to make a comparison")

for k, a in args_compare.items():
    if not isinstance(a, list):
        args_compare[k] = [a] * n_compare

Comparison will be done on the following parameters:
	behaviour: ['crawl', 'crawl', 'crawl', 'crawl', 'crawl']
	pretrained_behav: ['crawl', 'crawl', 'crawl', 'crawl', 'crawl']
	pretrained_model_folder: [False, True, True, True, False]
	n_gridcells: [0, 0, 0, 0, 25]
	gridcells_softmax: [False, False, False, False, True]
	gridcells_modules: [None, None, None, None, [0.2, 0.4, 0.6]]
	gridcells_orientations: [None, None, None, None, [0.1]]
	stride: [10, 20, 25, 30, 30]
	reset_hidden_at: [None, None, None, None, 10]


In [27]:
from utils.trainer import RNNTrainer

activity_dirs_roc = []
models_roc = []

print("Comparing the activity from the following directories:")
for i in range(n_compare):
    model_name = RNNTrainer.define_model_name({k: v[i] for k, v in args_compare.items()})
    
    env_shape = args_compare['env'][i].split('_')[0]
    trained_behav_list = args_compare['pretrained_behav'][i].split(',')
    behav = trained_behav_list.pop()
    if len(trained_behav_list)>0:
        folder_name = '_'.join(trained_behav_list)
    else:
        folder_name = "vanilla"
    exp_dir = os.path.join(
        DATA_DIR, env_shape, behav, "predictions", args_compare['env'][i],
        folder_name, model_name
    )

    activity_dir = f"act_{args_compare['behaviour'][i]}_epoch"
    if args_compare['epoch'][i] is not None:
        epoch = args_compare['epoch'][i]
    else:
        dirs = [d for d in os.listdir(exp_dir) if re.match(rf"{activity_dir}\d+", d)]
        epoch = max([int(re.findall(r'\d+', d)[-1]) for d in dirs])

    models_roc.append(
        torch.load(
            os.path.join(exp_dir, f"rnn_epoch{epoch}.pth"),
            weights_only=False,
            map_location=torch.device(DEVICE)
        ).to(DEVICE)
    )
    activity_dirs_roc.append(os.path.join(exp_dir, f"{activity_dir}{epoch}"))

    print(activity_dirs_roc[-1])

Comparing the activity from the following directories:
/media/data/vrtopc/box/crawl/predictions/box_messy/vanilla/RNN_f1_w9_st10_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_crawl_epoch1500
/media/data/vrtopc/box/crawl/predictions/box_messy/vanilla/RNN_ft_f1_w9_st20_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_crawl_epoch1500
/media/data/vrtopc/box/crawl/predictions/box_messy/vanilla/RNN_ft_f1_w9_st25_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_crawl_epoch1500
/media/data/vrtopc/box/crawl/predictions/box_messy/vanilla/RNN_ft_f1_w9_st30_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_crawl_epoch1500
/media/data/vrtopc/box/crawl/predictions/box_messy/vanilla/RNN_gridcellssm25_mod[0.2,0.4,0.6]_ori[0.1]_reset10_f1_w9_st30_fss4_do[0,0,0]_lat500_nlsigmoid_hreg0.0_wreg0.0_s01/act_crawl_epoch1500


# Extract values

### Metrics

In [28]:
sir_dict_real = {}
rvl_dict_real = {}
sid_dict_real = {}

for age in ages:
    for rat in data_dict_age[age].keys():
        # Determine dataset and rat key for matching in df_data
        dataset = 'science2010_data_struct' if rat in ratnames_science else 'muessig_data_struct'
        k = '_'.join(rat.split('_')[:2])
        
        # Get cluster index from df_data
        c = df_data[
            (df_data['dataset'] == dataset) &
            (df_data['age'] == age) &
            (df_data['rat'] == k)
        ]['cluster_idx_gm'].values
        
        if len(c) == 0:
            c = df_data[
                (df_data['dataset'] == dataset) &
                (df_data['age'] == age) &
                (df_data['rat'].str.contains(k.split('_')[0]))
            ]['cluster_idx_gm'].values
            
        if len(c) == 0 : continue

        assert len(np.unique(c)) == 1
        c = c[0]
        
        # Initialize cluster in data dictionary if not exists
        if c not in sir_dict_real.keys():
            sir_dict_real[c] = []
            rvl_dict_real[c] = []
            sid_dict_real[c] = []
        
        sir_dict_real[c].extend(list(data_dict_age[age][rat]['si_rm']))
        rvl_dict_real[c].extend(list(data_dict_age[age][rat]['rvl_pm']))
        sid_dict_real[c].extend(list(data_dict_age[age][rat]['si_pm']))

sir_real = []
rvl_real = []
sid_real = []
for c in sorted(sir_dict_real.keys()):
    sir_real.append(sir_dict_real[c])
    rvl_real.append(rvl_dict_real[c])
    sid_real.append(sid_dict_real[c])


In [29]:
sir_model_dev = []
rvl_model_dev = []
sid_model_dev = []

for ad in activity_dirs_dev[1:]:
    sir = np.load(os.path.join(ad, 'place', "si.npy"))
    rvl = np.load(os.path.join(ad, "hd", "rvl.npy"))
    sid = np.load(os.path.join(ad, "hd", "si.npy"))

    sir_model_dev.append(sir)
    rvl_model_dev.append(rvl)
    sid_model_dev.append(sid)

In [30]:
sir_model_roc = []
rvl_model_roc = []
sid_model_roc = []

for ad in activity_dirs_roc[1:]:
    sir_curr = np.load(os.path.join(ad, 'place', "si.npy"))
    sir_model_roc.append(np.nan_to_num(sir_curr, nan=np.nanmean(sir_curr)))
    rvl_curr = np.load(os.path.join(ad, "hd", "rvl.npy"))
    rvl_model_roc.append(np.nan_to_num(rvl_curr, nan=np.nanmean(rvl_curr)))
    sid_curr = np.load(os.path.join(ad, "hd", "si.npy"))
    sid_model_roc.append(np.nan_to_num(sid_curr, nan=np.nanmean(sid_curr)))

### Percentage of place, HD, place+HD cells

In [31]:
LATENT_DIM = 500

In [32]:
pc_perc_real_dict = {}
pc_n_real_dict = {}

phdc_perc_real_dict = {}
phdc_n_real_dict = {}

hdc_perc_real_dict = {}
hdc_n_real_dict = {}

n_real_dict = {}

for age in ages:
    for rat in data_dict_age[age].keys():
        dataset = 'science2010_data_struct' if rat in ratnames_science else 'muessig_data_struct'
        k = '_'.join(rat.split('_')[:2])
        c = df_data[
            (df_data['dataset'] == dataset) &
            (df_data['age'] == age) &
            (df_data['rat'] == k)
        ]['cluster_idx_gm'].values

        if len(c) == 0:
            c = df_data[
                (df_data['dataset'] == dataset) &
                (df_data['age'] == age) &
                (df_data['rat'].str.contains(k.split('_')[0]))
            ]['cluster_idx_gm'].values
            
        if len(c) == 0 : continue

        assert len(np.unique(c)) == 1
        c = c[0]

        if c not in pc_perc_real_dict.keys():
            pc_perc_real_dict[c] = []
            pc_n_real_dict[c] = []
            phdc_perc_real_dict[c] = []
            phdc_n_real_dict[c] = []
            hdc_perc_real_dict[c] = []
            hdc_n_real_dict[c] = []
            n_real_dict[c] = []

        selected_place_units = data_dict_age[age][rat]['selected_place_units']
        selected_hd_units = data_dict_age[age][rat]['selected_hd_units']
        selected_place_hd_units = data_dict_age[age][rat]['selected_place_hd_units']
        n_cells = len(data_dict_age[age][rat]['rate_maps'])
        n_real_dict[c].append(n_cells)

        n_pc = len(selected_place_units) + len(selected_place_hd_units)
        pc_n_real_dict[c].append(n_pc)
        pc_perc_real_dict[c].append(n_pc / n_cells)

        n_phdc = len(selected_place_hd_units)
        phdc_n_real_dict[c].append(n_phdc)
        phdc_perc_real_dict[c].append(n_phdc / n_cells)

        n_hdc = (len(selected_hd_units) + len(selected_place_hd_units))
        hdc_n_real_dict[c].append(n_hdc)
        hdc_perc_real_dict[c].append(n_hdc / n_cells)

pc_perc_real = []
pc_n_real = []

phdc_perc_real = []
phdc_n_real = []

hdc_perc_real = []
hdc_n_real = []

n_real = []
for c in sorted(pc_perc_real_dict.keys()):
    pc_perc_real.append(pc_perc_real_dict[c])
    pc_n_real.append(pc_n_real_dict[c])
    hdc_perc_real.append(hdc_perc_real_dict[c])
    hdc_n_real.append(hdc_n_real_dict[c])
    phdc_perc_real.append(phdc_perc_real_dict[c])
    phdc_n_real.append(phdc_n_real_dict[c])

    n_real.append(n_real_dict[c])

In [33]:
pc_perc_model_dev = []
phdc_perc_model_dev = []
hdc_perc_model_dev = []

for ad in activity_dirs_dev[1:]:
    selected_place_units = np.load(os.path.join(ad, "indices_place_cells.npy"))
    selected_place_hd_units = np.load(os.path.join(ad, "indices_conjunctive_cells.npy"))
    selected_hd_units = np.load(os.path.join(ad, "indices_hd_cells.npy"))

    pc_perc_model_dev.append(
        (len(selected_place_units)+len(selected_place_hd_units)) /
        LATENT_DIM
    )
    phdc_perc_model_dev.append(
        len(selected_place_hd_units) /
        LATENT_DIM
    )
    hdc_perc_model_dev.append(
        (len(selected_hd_units)+len(selected_place_hd_units)) /
        LATENT_DIM
    )

In [34]:
pc_perc_model_roc = []
phdc_perc_model_roc = []
hdc_perc_model_roc = []

for ad in activity_dirs_roc[1:]:
    selected_place_units = np.load(os.path.join(ad, "indices_place_cells.npy"))
    selected_place_hd_units = np.load(os.path.join(ad, "indices_conjunctive_cells.npy"))
    selected_hd_units = np.load(os.path.join(ad, "indices_hd_cells.npy"))

    pc_perc_model_roc.append(
        (len(selected_place_units)+len(selected_place_hd_units)) /
        LATENT_DIM
    )
    phdc_perc_model_roc.append(
        len(selected_place_hd_units) /
        LATENT_DIM
    )
    hdc_perc_model_roc.append(
        (len(selected_hd_units)+len(selected_place_hd_units)) /
        LATENT_DIM
    )

# Likelihood test

### Generate probability distributions

In [35]:
N_BINS = 100

In [36]:
def calculate_prob_distribs(metric_real, metric_model_dev, metric_model_roc, n_bins, c):
    metric_real_curr = metric_real[c]
    metric_model_dev_curr = metric_model_dev[c]
    metric_model_roc_curr = metric_model_roc[c]
    metric_min = min(np.min(metric_real_curr), np.min(metric_model_dev_curr), np.min(metric_model_roc_curr))
    metric_max = max(np.max(metric_real_curr), np.max(metric_model_dev_curr), np.max(metric_model_roc_curr))
    hist_dev, metric_edges_dev = np.histogram(metric_model_dev_curr, bins=n_bins, range=(metric_min, metric_max), density=False)
    hist_roc, metric_edges_roc = np.histogram(metric_model_roc_curr, bins=n_bins, range=(metric_min, metric_max), density=False)

    # convert histogram to probability distribution using softmax
    pd_dev = np.exp(hist_dev)/np.sum(np.exp(hist_dev))
    pd_roc = np.exp(hist_roc)/np.sum(np.exp(hist_roc))

    if not np.allclose(metric_edges_dev, metric_edges_roc):
        raise ValueError("Metric edges are not the same for dev and roc")
    if ~np.isclose(np.sum(pd_dev),1) or ~np.isclose(np.sum(pd_roc), 1):
        raise ValueError("Probability distribution does not sum to 1")
    
    return pd_dev, pd_roc, metric_edges_dev

def calculate_pdfs(metric_real, metric_model_dev, metric_model_roc, n_bins, c):
    metric_real_curr = metric_real[c]
    metric_model_dev_curr = metric_model_dev[c]
    metric_model_roc_curr = metric_model_roc[c]
    metric_min = min(np.min(metric_real_curr), np.min(metric_model_dev_curr), np.min(metric_model_roc_curr))
    metric_max = max(np.max(metric_real_curr), np.max(metric_model_dev_curr), np.max(metric_model_roc_curr))
    pdf_dev, metric_edges_dev = np.histogram(metric_model_dev_curr, bins=n_bins, range=(metric_min, metric_max), density=True)
    pdf_roc, metric_edges_roc = np.histogram(metric_model_roc_curr, bins=n_bins, range=(metric_min, metric_max), density=True)

    if not np.allclose(metric_edges_dev, metric_edges_roc):
        raise ValueError("Metric edges are not the same for dev and roc")
    
    return pdf_dev, pdf_roc, metric_edges_dev


In [37]:
if WITH_GRID_CELLS:
    sir_model_dev = sir_model_dev[:-2] + [sir_model_dev[-1]]
    rvl_model_dev = rvl_model_dev[:-2] + [rvl_model_dev[-1]]
    sid_model_dev = sid_model_dev[:-2] + [sid_model_dev[-1]]
    pc_perc_model_dev = pc_perc_model_dev[:-2] + [pc_perc_model_dev[-1]]
    phdc_perc_model_dev = phdc_perc_model_dev[:-2] + [phdc_perc_model_dev[-1]]
    hdc_perc_model_dev = hdc_perc_model_dev[:-2] + [hdc_perc_model_dev[-1]]

    sir_model_roc = sir_model_roc[:-2] + [sir_model_roc[-1]]
    rvl_model_roc = rvl_model_roc[:-2] + [rvl_model_roc[-1]]
    sid_model_roc = sid_model_roc[:-2] + [sid_model_roc[-1]]
    pc_perc_model_roc = pc_perc_model_roc[:-2] + [pc_perc_model_roc[-1]]
    phdc_perc_model_roc = phdc_perc_model_roc[:-2] + [phdc_perc_model_roc[-1]]
    hdc_perc_model_roc = hdc_perc_model_roc[:-2] + [hdc_perc_model_roc[-1]]

In [38]:
sir_pd_model_dev, rvl_pd_model_dev, sid_pd_model_dev = [], [], []
sir_pd_model_roc, rvl_pd_model_roc, sid_pd_model_roc = [], [], []
sir_edges, rvl_edges, sid_edges = [], [], []

for c in range(len(sir_real)):
    pdf_dev, pdf_roc, metric_edges = calculate_prob_distribs(
        sir_real, sir_model_dev, sir_model_roc, N_BINS, c
    )
    sir_pd_model_dev.append(pdf_dev)
    sir_pd_model_roc.append(pdf_roc)
    sir_edges.append(metric_edges)

    pdf_dev, pdf_roc, metric_edges = calculate_prob_distribs(
        rvl_real, rvl_model_dev, rvl_model_roc, N_BINS, c
    )
    rvl_pd_model_dev.append(pdf_dev)
    rvl_pd_model_roc.append(pdf_roc)
    rvl_edges.append(metric_edges)

    pdf_dev, pdf_roc, metric_edges = calculate_prob_distribs(
        sid_real, sid_model_dev, sid_model_roc, N_BINS, c
    )
    sid_pd_model_dev.append(pdf_dev)
    sid_pd_model_roc.append(pdf_roc)
    sid_edges.append(metric_edges)


### Calculate log-likelihoods

In [39]:
def calculate_loglikelihood(metric_real, pd_model_dev, pd_model_roc, metric_edges):
    bin_idx_real = np.digitize(metric_real, metric_edges)-1
    if -1 in bin_idx_real:
        raise ValueError("-1 should not be in bin_idx_real")
    bin_idx_real = np.clip(bin_idx_real, a_min=None, a_max=len(pd_model_dev)-1)

    ll_dev = np.log(pd_model_dev[bin_idx_real])
    ll_roc = np.log(pd_model_roc[bin_idx_real])

    perc_inf_dev = np.sum(ll_dev == -np.inf) / len(ll_dev) *100
    perc_inf_roc = np.sum(ll_roc == -np.inf) / len(ll_dev) *100

    return np.sum(ll_dev[ll_dev != -np.inf]), np.sum(ll_roc[ll_roc != -np.inf]), perc_inf_dev, perc_inf_roc

In [40]:
sir_ll_dev, sir_ll_roc = [], []
sir_pinf_dev, sir_pinf_roc = [], []
rvl_ll_dev, rvl_ll_roc = [], []
rvl_pinf_dev, rvl_pinf_roc = [], []
sid_ll_dev, sid_ll_roc = [], []
sid_pinf_dev, sid_pinf_roc = [], []


for c in range(len(sir_real)):
    ll_dev, ll_roc, pinf_dev, pinf_roc = calculate_loglikelihood(
        metric_real=sir_real[c],
        pd_model_dev=sir_pd_model_dev[c],
        pd_model_roc=sir_pd_model_roc[c],
        metric_edges=sir_edges[c]
    )
    sir_ll_dev.append(ll_dev)
    sir_ll_roc.append(ll_roc)
    sir_pinf_dev.append(pinf_dev)
    sir_pinf_roc.append(pinf_roc)

    ll_dev, ll_roc, pinf_dev, pinf_roc = calculate_loglikelihood(
        metric_real=rvl_real[c],
        pd_model_dev=rvl_pd_model_dev[c],
        pd_model_roc=rvl_pd_model_roc[c],
        metric_edges=rvl_edges[c]
    )
    rvl_ll_dev.append(ll_dev)
    rvl_ll_roc.append(ll_roc)
    rvl_pinf_dev.append(pinf_dev)
    rvl_pinf_roc.append(pinf_roc)

    ll_dev, ll_roc, pinf_dev, pinf_roc = calculate_loglikelihood(
        metric_real=sid_real[c],
        pd_model_dev=sid_pd_model_dev[c],
        pd_model_roc=sid_pd_model_roc[c],
        metric_edges=sid_edges[c]
    )
    sid_ll_dev.append(ll_dev)
    sid_ll_roc.append(ll_roc)
    sid_pinf_dev.append(pinf_dev)
    sid_pinf_roc.append(pinf_roc)

### Compare log-likelihoods

$SI_r$

In [41]:
len(sir_ll_roc)

3

In [42]:
ll_dev_tot = 0
ll_tot_roc = 0
for c in range(len(sir_real)):
    print(f'C={c}, LL dev={sir_ll_dev[c]:,.0f}, LL roc={sir_ll_roc[c]:,.0f}', end=' ')
    ll_dev_tot += sir_ll_dev[c]
    ll_tot_roc += sir_ll_roc[c]
    if sir_ll_dev[c] > sir_ll_roc[c]:
        print('!!! DEV WINS')
    else:
        print('??? ROC WINS')
print(f'TOT, LL dev={ll_dev_tot:,.0f}, LL roc={ll_tot_roc:,.0f}', end=' ')
if ll_dev_tot > ll_tot_roc:
    print('\t!!! DEV WINS')
else:
    print('\t??? ROC WINS')
print(flush=True)

C=0, LL dev=-11,731, LL roc=-12,490 !!! DEV WINS
C=1, LL dev=-30,930, LL roc=-48,815 !!! DEV WINS
C=2, LL dev=-9,028, LL roc=-16,235 !!! DEV WINS
TOT, LL dev=-51,689, LL roc=-77,540 	!!! DEV WINS



$RVL$

In [43]:

ll_dev_tot = 0
ll_tot_roc = 0
for c in range(len(sir_real)):
    print(f'C={c}, LL dev={rvl_ll_dev[c]:,.0f}, LL roc={rvl_ll_roc[c]:,.0f}', end=' ')
    ll_dev_tot += rvl_ll_dev[c]
    ll_tot_roc += rvl_ll_roc[c]
    if rvl_ll_dev[c] > rvl_ll_roc[c]:
        print('!!! DEV WINS')
    else:
        print('??? ROC WINS')
print(f'TOT, LL dev={ll_dev_tot:,.0f}, LL roc={ll_tot_roc:,.0f}', end=' ')
if ll_dev_tot > ll_tot_roc:
    print('\t!!! DEV WINS')
else:
    print('\t??? ROC WINS')
print(flush=True)

C=0, LL dev=-4,900, LL roc=-5,696 !!! DEV WINS
C=1, LL dev=-12,354, LL roc=-14,075 !!! DEV WINS
C=2, LL dev=-2,805, LL roc=-4,336 !!! DEV WINS
TOT, LL dev=-20,059, LL roc=-24,107 	!!! DEV WINS



$SI_d$

In [44]:

ll_dev_tot = 0
ll_tot_roc = 0
for c in range(len(sir_real)):
    print(f'C={c}, LL dev={sid_ll_dev[c]:,.0f}, LL roc={sid_ll_roc[c]:,.0f}', end=' ')
    ll_dev_tot += sid_ll_dev[c]
    ll_tot_roc += sid_ll_roc[c]
    if sid_ll_dev[c] > sid_ll_roc[c]:
        print('!!! DEV WINS')
    else:
        print('??? ROC WINS')
print(f'TOT, LL dev={ll_dev_tot:,.0f}, LL roc={ll_tot_roc:,.0f}', end=' ')
if ll_dev_tot > ll_tot_roc:
    print('\t!!! DEV WINS')
else:
    print('\t??? ROC WINS')
print(flush=True)

C=0, LL dev=-69,064, LL roc=-83,141 !!! DEV WINS
C=1, LL dev=-157,763, LL roc=-144,059 ??? ROC WINS
C=2, LL dev=-31,564, LL roc=-41,968 !!! DEV WINS
TOT, LL dev=-258,391, LL roc=-269,168 	!!! DEV WINS



### Calculate BIC

In [45]:
dof_dev = 4
dof_roc = 4

$SI_r$

In [46]:

ll_dev_tot = 0
ll_roc_tot = 0
n_tot = 0
for c in range(len(sir_real)):
    ll_dev = sir_ll_dev[c]
    ll_roc = sir_ll_roc[c]
    n = len(sir_model_dev[c])
    print(f'C={c}, L dev={ll_dev:,.0f}, L roc={ll_roc:,.0f}', end=',\t\t')
    ll_dev_tot += ll_dev
    ll_roc_tot += ll_roc
    n_tot += n
    
    bic_dev = -2*ll_dev + dof_dev*np.log(n)
    bic_roc = -2*ll_roc + dof_roc*np.log(n)
    delta_bic = bic_roc - bic_dev
    print(f'ΔBIC = {delta_bic:,.0f}')

print(f'TOT, LL dev={ll_dev_tot:,.0f}, LL roc={ll_roc_tot:,.0f}', end=',\t\t')

bic_dev = -2*ll_dev_tot + dof_dev*np.log(n_tot)
bic_roc = -2*ll_roc_tot + dof_roc*np.log(n_tot)
delta_bic = bic_roc - bic_dev
print(f'ΔBIC = {delta_bic:,.0f}')
print(flush=True)

C=0, L dev=-11,731, L roc=-12,490,		ΔBIC = 1,518
C=1, L dev=-30,930, L roc=-48,815,		ΔBIC = 35,770
C=2, L dev=-9,028, L roc=-16,235,		ΔBIC = 14,414
TOT, LL dev=-51,689, LL roc=-77,540,		ΔBIC = 51,702



$RVL$

In [47]:

ll_dev_tot = 0
ll_roc_tot = 0
n_tot = 0
for c in range(len(rvl_real)):
    ll_dev = rvl_ll_dev[c]
    ll_roc = rvl_ll_roc[c]
    n = len(rvl_model_dev[c])
    print(f'C={c}, L dev={ll_dev:,.0f}, L roc={ll_roc:,.0f}', end=',\t\t')
    ll_dev_tot += ll_dev
    ll_roc_tot += ll_roc
    n_tot += n
    
    bic_dev = -2*ll_dev + dof_dev*np.log(n)
    bic_roc = -2*ll_roc + dof_roc*np.log(n)
    delta_bic = bic_roc - bic_dev
    print(f'ΔBIC = {delta_bic:,.0f}')

print(f'TOT, LL dev={ll_dev_tot:,.0f}, LL roc={ll_roc_tot:,.0f}', end=',\t\t')

bic_dev = -2*ll_dev_tot + dof_dev*np.log(n_tot)
bic_roc = -2*ll_roc_tot + dof_roc*np.log(n_tot)
delta_bic = bic_roc - bic_dev
print(f'ΔBIC = {delta_bic:,.0f}')
print(flush=True)

C=0, L dev=-4,900, L roc=-5,696,		ΔBIC = 1,592
C=1, L dev=-12,354, L roc=-14,075,		ΔBIC = 3,442
C=2, L dev=-2,805, L roc=-4,336,		ΔBIC = 3,062
TOT, LL dev=-20,059, LL roc=-24,107,		ΔBIC = 8,096



$SI_d$

In [48]:

ll_dev_tot = 0
ll_roc_tot = 0
n_tot = 0
for c in range(len(sid_real)):
    ll_dev = sid_ll_dev[c]
    ll_roc = sid_ll_roc[c]
    n = len(sid_model_dev[c])
    print(f'C={c}, L dev={ll_dev:,.0f}, L roc={ll_roc:,.0f}', end=',\t\t')
    ll_dev_tot += ll_dev
    ll_roc_tot += ll_roc
    n_tot += n
    
    bic_dev = -2*ll_dev + dof_dev*np.log(n)
    bic_roc = -2*ll_roc + dof_roc*np.log(n)
    delta_bic = bic_roc - bic_dev
    print(f'ΔBIC = {delta_bic:,.0f}')

print(f'TOT, LL dev={ll_dev_tot:,.0f}, LL roc={ll_roc_tot:,.0f}', end=',\t\t')

bic_dev = -2*ll_dev_tot + dof_dev*np.log(n_tot)
bic_roc = -2*ll_roc_tot + dof_roc*np.log(n_tot)
delta_bic = bic_roc - bic_dev
print(f'ΔBIC = {delta_bic:,.0f}')
print(flush=True)

C=0, L dev=-69,064, L roc=-83,141,		ΔBIC = 28,154
C=1, L dev=-157,763, L roc=-144,059,		ΔBIC = -27,408
C=2, L dev=-31,564, L roc=-41,968,		ΔBIC = 20,808
TOT, LL dev=-258,391, LL roc=-269,168,		ΔBIC = 21,554



# Binomial log-likelihood

In [49]:
from scipy.special import comb

def calculate_binomial_loglikelihood(cell_perc_model, cell_n_real, tot_n_cells_real):
    return np.sum(
        np.log(comb(tot_n_cells_real, cell_n_real)) +
        cell_n_real*np.log(cell_perc_model) +
        (tot_n_cells_real - cell_n_real)*np.log(1-cell_perc_model)
    )

In [50]:
pc_bll_dev, pc_bll_roc = [], []
hdc_bll_dev, hdc_bll_roc = [], []
phdc_bll_dev, phdc_bll_roc = [], []

for c in range(len(pc_perc_real)):
    pc_bll_dev.append(calculate_binomial_loglikelihood(
        pc_perc_model_dev[c], np.array(pc_n_real[c]), np.array(n_real[c])
    ))
    pc_bll_roc.append(calculate_binomial_loglikelihood(
        pc_perc_model_roc[c], np.array(pc_n_real[c]), np.array(n_real[c])
    ))

    hdc_bll_dev.append(calculate_binomial_loglikelihood(
        hdc_perc_model_dev[c], np.array(hdc_n_real[c]), np.array(n_real[c])
    ))
    hdc_bll_roc.append(calculate_binomial_loglikelihood(
        hdc_perc_model_roc[c], np.array(hdc_n_real[c]), np.array(n_real[c])
    ))

    phdc_bll_dev.append(calculate_binomial_loglikelihood(
        phdc_perc_model_dev[c], np.array(phdc_n_real[c]), np.array(n_real[c])
    ))
    phdc_bll_roc.append(calculate_binomial_loglikelihood(
        phdc_perc_model_roc[c], np.array(phdc_n_real[c]), np.array(n_real[c])
    ))


### Compare bin log-likelihoods

Perc. Place cells

In [51]:
bll_dev_tot = 0
bll_tot_roc = 0
for c in range(len(pc_perc_real)):
    bll_dev = pc_bll_dev[c]
    bll_roc = pc_bll_roc[c]
    print(f'C={c}, BLL dev={bll_dev:,.0f}, BLL roc={bll_roc:,.0f}', end=' ')
    bll_dev_tot += bll_dev
    bll_tot_roc += bll_roc
    if bll_dev > bll_roc:
        print('!!! DEV WINS')
    else:
        print('??? ROC WINS')
print(f'TOT, LL dev={bll_dev_tot:,.0f}, LL roc={bll_tot_roc:,.0f}', end=' ')
if bll_dev_tot > bll_tot_roc:
    print('\t!!! DEV WINS')
else:
    print('\t??? ROC WINS')
print(flush=True)

C=0, BLL dev=-83, BLL roc=-130 !!! DEV WINS
C=1, BLL dev=-294, BLL roc=-471 !!! DEV WINS
C=2, BLL dev=-101, BLL roc=-186 !!! DEV WINS
TOT, LL dev=-478, LL roc=-787 	!!! DEV WINS



Perc. HD cells

In [52]:
bll_dev_tot = 0
bll_tot_roc = 0
for c in range(len(pc_perc_real)):
    bll_dev = hdc_bll_dev[c]
    bll_roc = hdc_bll_roc[c]
    print(f'C={c}, BLL dev={bll_dev:,.0f}, BLL roc={bll_roc:,.0f}', end=' ')
    bll_dev_tot += bll_dev
    bll_tot_roc += bll_roc
    if bll_dev > bll_roc:
        print('!!! DEV WINS')
    else:
        print('??? ROC WINS')
print(f'TOT, LL dev={bll_dev_tot:,.0f}, LL roc={bll_tot_roc:,.0f}', end=' ')
if bll_dev_tot > bll_tot_roc:
    print('\t!!! DEV WINS')
else:
    print('\t??? ROC WINS')
print(flush=True)

C=0, BLL dev=-51, BLL roc=-78 !!! DEV WINS
C=1, BLL dev=-173, BLL roc=-254 !!! DEV WINS
C=2, BLL dev=-21, BLL roc=-30 !!! DEV WINS
TOT, LL dev=-244, LL roc=-362 	!!! DEV WINS



Perc. Place+HD cells

In [53]:
bll_dev_tot = 0
bll_tot_roc = 0
for c in range(len(pc_perc_real)):
    bll_dev = phdc_bll_dev[c]
    bll_roc = phdc_bll_roc[c]
    print(f'C={c}, BLL dev={bll_dev:,.0f}, BLL roc={bll_roc:,.0f}', end=' ')
    bll_dev_tot += bll_dev
    bll_tot_roc += bll_roc
    if bll_dev > bll_roc:
        print('!!! DEV WINS')
    else:
        print('??? ROC WINS')
print(f'TOT, LL dev={bll_dev_tot:,.0f}, LL roc={bll_tot_roc:,.0f}', end=' ')
if bll_dev_tot > bll_tot_roc:
    print('\t!!! DEV WINS')
else:
    print('\t??? ROC WINS')
print(flush=True)

C=0, BLL dev=-52, BLL roc=-81 !!! DEV WINS
C=1, BLL dev=-185, BLL roc=-334 !!! DEV WINS
C=2, BLL dev=-19, BLL roc=-46 !!! DEV WINS
TOT, LL dev=-257, LL roc=-462 	!!! DEV WINS



### Calculate BIC

In [54]:
dof_dev = 4
dof_roc = 4

Perc. Place cells

In [55]:

bll_dev_tot = 0
bll_roc_tot = 0
n_tot = 0
for c in range(len(pc_perc_real)):
    bll_dev = pc_bll_dev[c]
    bll_roc = pc_bll_roc[c]
    n = len(sir_real[c])
    print(f'C={c}, L dev={bll_dev:,.0f}, L roc={bll_roc:,.0f}', end=',\t\t')
    bll_dev_tot += bll_dev
    bll_roc_tot += bll_roc
    n_tot += n
    
    bic_dev = -2*bll_dev + dof_dev*np.log(n)
    bic_roc = -2*bll_roc + dof_roc*np.log(n)
    delta_bic = bic_roc - bic_dev
    print(f'ΔBIC = {delta_bic:,.0f}')

print(f'TOT, LL dev={bll_dev_tot:,.0f}, LL roc={bll_roc_tot:,.0f}', end=',\t\t')

bic_dev = -2*bll_dev_tot + dof_dev*np.log(n_tot)
bic_roc = -2*bll_roc_tot + dof_roc*np.log(n_tot)
delta_bic = bic_roc - bic_dev
print(f'ΔBIC = {delta_bic:,.0f}')
print(flush=True)

C=0, L dev=-83, L roc=-130,		ΔBIC = 94
C=1, L dev=-294, L roc=-471,		ΔBIC = 354
C=2, L dev=-101, L roc=-186,		ΔBIC = 170
TOT, LL dev=-478, LL roc=-787,		ΔBIC = 618



Perc. Place cells

In [56]:

bll_dev_tot = 0
bll_roc_tot = 0
n_tot = 0
for c in range(len(hdc_perc_real)):
    bll_dev = hdc_bll_dev[c]
    bll_roc = hdc_bll_roc[c]
    n = len(sir_real[c])
    print(f'C={c}, L dev={bll_dev:,.0f}, L roc={bll_roc:,.0f}', end=',\t\t')
    bll_dev_tot += bll_dev
    bll_roc_tot += bll_roc
    n_tot += n
    
    bic_dev = -2*bll_dev + dof_dev*np.log(n)
    bic_roc = -2*bll_roc + dof_roc*np.log(n)
    delta_bic = bic_roc - bic_dev
    print(f'ΔBIC = {delta_bic:,.0f}')

print(f'TOT, LL dev={bll_dev_tot:,.0f}, LL roc={bll_roc_tot:,.0f}', end=',\t\t')

bic_dev = -2*bll_dev_tot + dof_dev*np.log(n_tot)
bic_roc = -2*bll_roc_tot + dof_roc*np.log(n_tot)
delta_bic = bic_roc - bic_dev
print(f'ΔBIC = {delta_bic:,.0f}')
print(flush=True)

C=0, L dev=-51, L roc=-78,		ΔBIC = 54
C=1, L dev=-173, L roc=-254,		ΔBIC = 162
C=2, L dev=-21, L roc=-30,		ΔBIC = 19
TOT, LL dev=-244, LL roc=-362,		ΔBIC = 235



Perc. Place+HD cells

In [57]:

bll_dev_tot = 0
bll_roc_tot = 0
n_tot = 0
for c in range(len(phdc_perc_real)):
    bll_dev = phdc_bll_dev[c]
    bll_roc = phdc_bll_roc[c]
    n = len(sir_real[c])
    print(f'C={c}, L dev={bll_dev:,.0f}, L roc={bll_roc:,.0f}', end=',\t\t')
    bll_dev_tot += bll_dev
    bll_roc_tot += bll_roc
    n_tot += n
    
    bic_dev = -2*bll_dev + dof_dev*np.log(n)
    bic_roc = -2*bll_roc + dof_roc*np.log(n)
    delta_bic = bic_roc - bic_dev
    print(f'ΔBIC = {delta_bic:,.0f}')

print(f'TOT, LL dev={bll_dev_tot:,.0f}, LL roc={bll_roc_tot:,.0f}', end=',\t\t')

bic_dev = -2*bll_dev_tot + dof_dev*np.log(n_tot)
bic_roc = -2*bll_roc_tot + dof_roc*np.log(n_tot)
delta_bic = bic_roc - bic_dev
print(f'ΔBIC = {delta_bic:,.0f}')
print(flush=True)

C=0, L dev=-52, L roc=-81,		ΔBIC = 59
C=1, L dev=-185, L roc=-334,		ΔBIC = 298
C=2, L dev=-19, L roc=-46,		ΔBIC = 53
TOT, LL dev=-257, LL roc=-462,		ΔBIC = 410

